# TableGPT-R1 — Kaggle demo

Faithful implementation study of [TableGPT-R1](https://arxiv.org/abs/2512.20312):
an SFT warm-up followed by one RL stage on a deterministic synthetic tabular
task, using a small QLoRA policy.

**Not** paper-parity — full 8B RL needs A100-class compute (see
`docs/COMPUTE_AND_SCALING.md`).

**Setup:** Accelerator = GPU T4 x2, Internet = On. Upload the repo as a Kaggle
Dataset named `tablegpt-r1`.

In [ ]:
# 1. Dependencies (run once; ~1-2 min)
!pip install -q -U transformers datasets accelerate peft bitsandbytes matplotlib datasketch rouge-score

In [ ]:
# 2. Verify the environment
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0),
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nvidia-smi

In [ ]:
# 3. Locate the project (upload it as a Kaggle Dataset named 'tablegpt-r1')
import os, shutil, sys, glob

SRC = None
for cand in glob.glob('/kaggle/input/**/tablegpt_r1', recursive=True):
    if os.path.exists(os.path.join(cand, 'main.py')) and os.path.exists(os.path.join(cand, 'config.py')):
        SRC = cand
        break

WORK = '/kaggle/working/tablegpt_r1'
if SRC and not os.path.isdir(WORK):
    shutil.copytree(SRC, WORK)
    print('copied', SRC, '->', WORK)
assert os.path.isdir(WORK), 'repo not found: upload it as a Kaggle Dataset'
os.chdir(WORK)
sys.path.insert(0, WORK)
print('cwd', os.getcwd())

In [ ]:
# 4. Offline smoke test (no model required)
!python main.py pipeline

In [ ]:
# 5. Train: QLoRA SFT warm-up -> 1 RL stage
# T4 (Turing) has no bf16 -> code auto-selects fp16 for 4-bit compute + AMP.
# SFT now trains on the completion only (prompt masked); RL uses shaped rewards.
!python kaggle/train_kaggle.py \
    --policy-size 3b --qlora --task filter_aggregate \
    --sft-samples 256 --sft-epochs 3 --rl-steps 40 --group-size 8 \
    --max-new-tokens 128 \
    --out /kaggle/working/tablegpt_demo

In [ ]:
# 6. Plot the curves inline
import json, matplotlib.pyplot as plt
h = json.load(open('/kaggle/working/tablegpt_demo/metrics.json'))
steps = [r['step'] for r in h]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(steps, [r['mean_reward'] for r in h]); ax[0].set_title('mean reward')
ax[1].plot(steps, [r['loss'] for r in h]); ax[1].set_title('loss')
ax[2].plot(steps, [r['mean_entropy'] for r in h]); ax[2].set_title('mean entropy')
for a in ax: a.set_xlabel('RL step')
plt.show()

### 6b. Before/after evaluation

Compare the base policy against the trained LoRA adapter on a held-out
synthetic set (disjoint from training).

In [ ]:
# 6b. Before/after eval: base policy vs trained adapter
!python main.py eval-tabular \
    --model Qwen/Qwen2.5-3B --qlora --task filter_aggregate \
    --adapter /kaggle/working/tablegpt_demo/rl_adapter \
    --n 32 --seed 123 --max-new-tokens 128 \
    --json /kaggle/working/tablegpt_demo/eval_report.json

In [ ]:
# 7. Persist artifacts (download from the Kaggle output pane)
import os, shutil
OUT = '/kaggle/working/tablegpt_demo'
print(sorted(os.listdir(OUT)))
shutil.make_archive('/kaggle/working/tablegpt_demo_artifacts', 'zip', OUT)
print('wrote /kaggle/working/tablegpt_demo_artifacts.zip')